# Founder haplotypes

`xftsim` simulations can be highly realistic, but they can also be
quite simplistic. We can use real phased haplotype data or simulate
haplotypes as independent Bernoulli trials. As fully synthetic data is
extremely convenient to work with, we recommend at the very least
using such data to prototype or debug simulations.

In what follows we first introduce tools for generating haplotypes
from scratch, then tools for importing external haplotype data.

## Haplotypes from scratch

The simplest founder constructors generate independent Bernoulli
draws. Given a vector of per-locus allele frequencies,
`founders.founder_haplotypes_from_AFs()` builds a
`DenseHaplotypeArray`:


In [ ]:
import xftsim as xft
from xftsim import founders

afs = [0.0, 1.0, 0.5]
founders.founder_haplotypes_from_AFs(n=10, afs=afs)


For convenience, `founders.founder_haplotypes_uniform_AFs()` will
uniformly sample `m` allele frequencies between `minMAF` and
`1 - minMAF`:


In [ ]:
founders.founder_haplotypes_uniform_AFs(n=10, m=10, minMAF=0.05)

:::{note}
When founder constructors create variant metadata without an
explicit chromosome layout, `xftsim` evenly divides variants between
(up to) 22 chromosomes.
:::

Of course you can construct a `DenseHaplotypeArray` directly from a
3-D `(n, m, 2)` numpy array:


In [ ]:
import numpy as np

genos = np.random.binomial(1, 0.3, size=(100, 200, 2)).astype(np.int8)
xft.struct.DenseHaplotypeArray(genos)


## Haplotypes from external datasets

`xftsim` currently supports PLINK binary (bfile) and VCF/sgkit
formats. (We hope to add bgen / plink2 pfile support in the future.)

### PLINK bfiles

:::{warning}
The PLINK bfile format is inherently diploid and will break phasing —
haplotypes at heterozygous loci are assigned randomly. For phased
data, prefer the VCF / sgkit path or a GRG (see below).
:::

We can read a bfile using
`founders.founder_haplotypes_from_plink_bfile()`. Example using the
data packaged with `pandas_plink`:


In [ ]:
from pandas_plink import get_data_folder
from os.path import join

pdat = founders.founder_haplotypes_from_plink_bfile(
    join(get_data_folder(), 'chr*.bed')
)
pdat


### GRG (graph-based representation)

`xftsim` supports [GRG](https://github.com/aprilweilab/grg)
graphs for memory-efficient haplotype storage. Two helpers generate
GRG-backed founders from coalescent simulators:

- `xftsim.founders.founder_haplotypes_from_msprime_grg()` — simulates
  via msprime, converts the tree sequence to a GRG, and returns a
  `GraphHaplotypeOperator`.
- `xftsim.founders.founder_haplotypes_from_stdpopsim_grg()` — simulates
  via a stdpopsim demographic model (e.g. `OutOfAfrica_3G09`), converts
  to GRG, and returns a `GraphHaplotypeOperator` with per-individual
  population labels and real `pos_cM` from the contig's recombination
  map. Samples are specified per population (e.g.
  `{"YRI": 100, "CEU": 100}`).

You can also load an existing GRG file with `xftsim.io.load_grg()`.
All three return a `GraphHaplotypeOperator` that implements the same
`matvec` / `rmatvec` / `standardized_matvec` / `meiosis` API as
`DenseHaplotypeArray`, so it drops straight into a simulation.

GRG-native recombination means offspring remain `GraphHaplotypeOperator`
across all generations — the GRG compression advantage persists through
the entire simulation without dense materialization. See the
[GRG-Backed Genotypes example](../examples/07_grg_genotypes.ipynb) for
a detailed walkthrough.

Install GRG dependencies with `pip install xftsim[grg]` (includes
`pygrgl`, `msprime`, `tskit`, and `stdpopsim`).

## Haplotypes from npz

The preferred on-disk format for saving and loading haplotype arrays
in v0.9 is npz. Save with `xft.io.save_haplotypes_npz()` and load with
`xft.io.load_haplotypes_npz()` — both round-trip the genotypes plus the
full `SampleMeta` / `VariantMeta`.